In [0]:
%sql
SELECT *
FROM pokemon.bronze.pokemon_raw
limit 10;

In [0]:
%sql
SELECT
    COUNT(*) AS bronze_rows,
    COUNT(
        DISTINCT get_json_object(raw_payload, '$.id')
    ) AS distinct_pokemon_ids,
    COUNT_IF(
        get_json_object(raw_payload, '$.id') IS NULL
    ) AS missing_pokemon_ids
FROM pokemon.bronze.pokemon_raw
WHERE status_code = 200;

In [0]:
%sql
SELECT
    get_json_object(raw_payload, '$.id') AS pokemon_id,
    COUNT(*) AS record_count
FROM pokemon.bronze.pokemon_raw
WHERE status_code = 200
GROUP BY get_json_object(raw_payload, '$.id')
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    get_json_object(raw_payload, '$.id') AS pokemon_id,
    COUNT(*) AS record_count,
    COUNT(DISTINCT SHA2(raw_payload, 256)) AS distinct_payloads,
    MIN(_ingestion_timestamp) AS first_ingested_at,
    MAX(_ingestion_timestamp) AS last_ingested_at
FROM pokemon.bronze.pokemon_raw
WHERE status_code = 200
GROUP BY get_json_object(raw_payload, '$.id')
HAVING COUNT(*) > 1
ORDER BY CAST(
    get_json_object(raw_payload, '$.id') AS INT
)
LIMIT 20;

In [0]:
%sql
SELECT
    get_json_object(raw_payload, '$.id') AS pokemon_id,
    get_json_object(raw_payload, '$.name') AS name,
    get_json_object(raw_payload, '$.base_experience') AS base_experience,
    get_json_object(raw_payload, '$.height') AS height,
    get_json_object(raw_payload, '$.weight') AS weight,
    get_json_object(raw_payload, '$.types') AS types,
    _ingestion_timestamp,
    SHA2(raw_payload, 256) AS payload_hash
FROM pokemon.bronze.pokemon_raw
WHERE status_code = 200
  AND get_json_object(raw_payload, '$.id') = '1'
ORDER BY _ingestion_timestamp;

In [0]:
%sql
WITH versions AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            ORDER BY _ingestion_timestamp ASC
        ) AS version_num
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
      AND get_json_object(raw_payload, '$.id') = '1'
),
comparison AS (
    SELECT
        MAX(CASE WHEN version_num = 1
            THEN raw_payload END) AS old_payload,
        MAX(CASE WHEN version_num = 2
            THEN raw_payload END) AS new_payload
    FROM versions
)
SELECT
    old_payload,
    new_payload
FROM comparison;

In [0]:
%sql
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
pokemon_types AS (
    SELECT
        get_json_object(raw_payload, '$.id') AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.types'),
            'ARRAY<STRUCT<
                slot: INT,
                type: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS types
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS pokemon_count,
    SUM(CASE WHEN types IS NULL THEN 1 ELSE 0 END)
        AS null_type_arrays,
    SUM(CASE WHEN SIZE(types) = 0 THEN 1 ELSE 0 END)
        AS empty_type_arrays,
    MIN(SIZE(types)) AS min_types_per_pokemon,
    MAX(SIZE(types)) AS max_types_per_pokemon
FROM pokemon_types;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.types'),
            'ARRAY<STRUCT<slot: INT, type: STRUCT<name: STRING, url: STRING>>>'
        ) AS types
    FROM ranked
    WHERE rn = 1
),
exploded AS (
    SELECT
        p.pokemon_id,
        type_entry.slot AS slot,
        type_entry.type.name AS type_name,
        type_entry.type.url AS type_url
    FROM parsed p
    LATERAL VIEW EXPLODE(p.types) exploded_types AS type_entry
)
SELECT
    COUNT(*) AS type_assignment_rows,
    COUNT(DISTINCT CONCAT(
        CAST(pokemon_id AS STRING), '-', CAST(slot AS STRING)
    )) AS distinct_pokemon_slot_keys,
    COUNT(DISTINCT CONCAT(
        CAST(pokemon_id AS STRING), '-', type_name
    )) AS distinct_pokemon_type_pairs
FROM exploded;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.abilities'),
            'ARRAY<STRUCT<
                is_hidden: BOOLEAN,
                slot: INT,
                ability: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS abilities
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS pokemon_count,
    SUM(CASE WHEN abilities IS NULL THEN 1 ELSE 0 END)
        AS null_ability_arrays,
    SUM(CASE WHEN SIZE(abilities) = 0 THEN 1 ELSE 0 END)
        AS empty_ability_arrays,
    MIN(SIZE(abilities)) AS min_abilities_per_pokemon,
    MAX(SIZE(abilities)) AS max_abilities_per_pokemon
FROM parsed;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.abilities'),
            'ARRAY<STRUCT<
                is_hidden: BOOLEAN,
                slot: INT,
                ability: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS abilities
    FROM ranked
    WHERE rn = 1
),
exploded AS (
    SELECT
        p.pokemon_id,
        ability_entry.slot AS slot,
        ability_entry.is_hidden AS is_hidden,
        ability_entry.ability.name AS ability_name,
        ability_entry.ability.url AS ability_url
    FROM parsed p
    LATERAL VIEW EXPLODE(p.abilities)
        exploded_abilities AS ability_entry
)
SELECT
    COUNT(*) AS ability_assignment_rows,
    COUNT(DISTINCT CONCAT(
        CAST(pokemon_id AS STRING), '-', CAST(slot AS STRING)
    )) AS distinct_pokemon_slot_keys,
    COUNT(DISTINCT CONCAT(
        CAST(pokemon_id AS STRING), '-', ability_name
    )) AS distinct_pokemon_ability_pairs,
    SUM(CASE WHEN slot IS NULL THEN 1 ELSE 0 END) AS null_slots,
    SUM(CASE WHEN is_hidden IS NULL THEN 1 ELSE 0 END)
        AS null_hidden_flags,
    SUM(CASE WHEN ability_name IS NULL THEN 1 ELSE 0 END)
        AS null_ability_names,
    SUM(CASE WHEN ability_url IS NULL THEN 1 ELSE 0 END)
        AS null_ability_urls
FROM exploded;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.stats'),
            'ARRAY<STRUCT<
                base_stat: INT,
                effort: INT,
                stat: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS stats
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS pokemon_count,
    SUM(CASE WHEN stats IS NULL THEN 1 ELSE 0 END)
        AS null_stat_arrays,
    SUM(CASE WHEN SIZE(stats) = 0 THEN 1 ELSE 0 END)
        AS empty_stat_arrays,
    MIN(SIZE(stats)) AS min_stats_per_pokemon,
    MAX(SIZE(stats)) AS max_stats_per_pokemon
FROM parsed;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.stats'),
            'ARRAY<STRUCT<
                base_stat: INT,
                effort: INT,
                stat: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS stats
    FROM ranked
    WHERE rn = 1
),
exploded AS (
    SELECT
        p.pokemon_id,
        stat_entry.stat.name AS stat_name,
        stat_entry.stat.url AS stat_url,
        stat_entry.base_stat AS base_stat,
        stat_entry.effort AS effort
    FROM parsed p
    LATERAL VIEW EXPLODE(p.stats)
        exploded_stats AS stat_entry
)
SELECT
    COUNT(*) AS stat_assignment_rows,
    COUNT(DISTINCT CONCAT(
        CAST(pokemon_id AS STRING), '-', stat_name
    )) AS distinct_pokemon_stat_keys,
    SUM(CASE WHEN stat_name IS NULL THEN 1 ELSE 0 END)
        AS null_stat_names,
    SUM(CASE WHEN stat_url IS NULL THEN 1 ELSE 0 END)
        AS null_stat_urls,
    SUM(CASE WHEN base_stat IS NULL THEN 1 ELSE 0 END)
        AS null_base_stats,
    SUM(CASE WHEN effort IS NULL THEN 1 ELSE 0 END)
        AS null_effort_values
FROM exploded;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.moves'),
            'ARRAY<STRUCT<
                move: STRUCT<name: STRING, url: STRING>,
                version_group_details: ARRAY<STRUCT<
                    level_learned_at: INT,
                    order: INT,
                    version_group: STRUCT<name: STRING, url: STRING>,
                    move_learn_method: STRUCT<name: STRING, url: STRING>
                >>
            >>'
        ) AS moves
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS pokemon_count,
    SUM(CASE WHEN moves IS NULL THEN 1 ELSE 0 END)
        AS null_move_arrays,
    SUM(CASE WHEN SIZE(moves) = 0 THEN 1 ELSE 0 END)
        AS empty_move_arrays,
    MIN(SIZE(moves)) AS min_moves_per_pokemon,
    MAX(SIZE(moves)) AS max_moves_per_pokemon,
    ROUND(AVG(SIZE(moves)), 2) AS avg_moves_per_pokemon
FROM parsed;

In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.moves'),
            'ARRAY<STRUCT<
                move: STRUCT<name: STRING, url: STRING>,
                version_group_details: ARRAY<STRUCT<
                    level_learned_at: INT,
                    order: INT,
                    version_group: STRUCT<name: STRING, url: STRING>,
                    move_learn_method: STRUCT<name: STRING, url: STRING>
                >>
            >>'
        ) AS moves
    FROM ranked
    WHERE rn = 1
),
exploded_moves AS (
    SELECT
        p.pokemon_id,
        move_entry.move.name AS move_name,
        move_entry.move.url AS move_url,
        move_entry.version_group_details AS version_group_details
    FROM parsed p
    LATERAL VIEW EXPLODE(p.moves)
        exploded_moves_table AS move_entry
)
SELECT
    COUNT(*) AS pokemon_move_rows,
    SUM(
        CASE WHEN version_group_details IS NULL THEN 1 ELSE 0 END
    ) AS null_detail_arrays,
    SUM(
        CASE WHEN SIZE(version_group_details) = 0 THEN 1 ELSE 0 END
    ) AS empty_detail_arrays,
    MIN(SIZE(version_group_details)) AS min_details_per_move,
    MAX(SIZE(version_group_details)) AS max_details_per_move,
    ROUND(AVG(SIZE(version_group_details)), 2)
        AS avg_details_per_move
FROM exploded_moves;

In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.moves'),
            'ARRAY<STRUCT<
                move: STRUCT<name: STRING, url: STRING>,
                version_group_details: ARRAY<STRUCT<
                    level_learned_at: INT,
                    order: INT,
                    version_group: STRUCT<name: STRING, url: STRING>,
                    move_learn_method: STRUCT<name: STRING, url: STRING>
                >>
            >>'
        ) AS moves
    FROM ranked
    WHERE rn = 1
),
exploded_moves AS (
    SELECT
        p.pokemon_id,
        m.move.name AS move_name,
        m.move.url AS move_url,
        m.version_group_details AS details
    FROM parsed p
    LATERAL VIEW EXPLODE(p.moves) move_lv AS m
),
exploded_details AS (
    SELECT
        m.pokemon_id,
        m.move_name,
        m.move_url,
        d.version_group.name AS version_group_name,
        d.move_learn_method.name AS learn_method_name,
        d.level_learned_at AS level_learned_at,
        d.order AS learn_order
    FROM exploded_moves m
    LATERAL VIEW EXPLODE(m.details) detail_lv AS d
)
SELECT
    COUNT(*) AS learning_detail_rows,

    COUNT(DISTINCT CONCAT(
        CAST(pokemon_id AS STRING), '|', move_name, '|',
        COALESCE(version_group_name, '<NULL>'), '|',
        COALESCE(learn_method_name, '<NULL>')
    )) AS distinct_candidate_keys,

    SUM(CASE WHEN move_name IS NULL THEN 1 ELSE 0 END)
        AS null_move_names,
    SUM(CASE WHEN version_group_name IS NULL THEN 1 ELSE 0 END)
        AS null_version_groups,
    SUM(CASE WHEN learn_method_name IS NULL THEN 1 ELSE 0 END)
        AS null_learn_methods,
    SUM(CASE WHEN level_learned_at IS NULL THEN 1 ELSE 0 END)
        AS null_levels,
    SUM(CASE WHEN learn_order IS NULL THEN 1 ELSE 0 END)
        AS null_orders
FROM exploded_details;


In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.moves'),
            'ARRAY<STRUCT<
                move: STRUCT<name: STRING, url: STRING>,
                version_group_details: ARRAY<STRUCT<
                    level_learned_at: INT,
                    order: INT,
                    version_group: STRUCT<name: STRING, url: STRING>,
                    move_learn_method: STRUCT<name: STRING, url: STRING>
                >>
            >>'
        ) AS moves
    FROM ranked
    WHERE rn = 1
),
exploded_moves AS (
    SELECT
        p.pokemon_id,
        m.move.name AS move_name,
        m.version_group_details AS details
    FROM parsed p
    LATERAL VIEW EXPLODE(p.moves) move_lv AS m
),
exploded_details AS (
    SELECT
        m.pokemon_id,
        m.move_name,
        d.version_group.name AS version_group_name,
        d.move_learn_method.name AS learn_method_name,
        d.level_learned_at,
        d.order AS learn_order
    FROM exploded_moves m
    LATERAL VIEW EXPLODE(m.details) detail_lv AS d
),
duplicates AS (
    SELECT
        pokemon_id,
        move_name,
        version_group_name,
        learn_method_name,
        COUNT(*) AS duplicate_count,
        COUNT(DISTINCT NAMED_STRUCT(
            'level', level_learned_at,
            'order', learn_order
        )) AS distinct_detail_variants,
        COLLECT_SET(NAMED_STRUCT(
            'level', level_learned_at,
            'order', learn_order
        )) AS detail_variants
    FROM exploded_details
    GROUP BY
        pokemon_id,
        move_name,
        version_group_name,
        learn_method_name
    HAVING COUNT(*) > 1
)
SELECT *
FROM duplicates
ORDER BY duplicate_count DESC
LIMIT 50;


In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.moves'),
            'ARRAY<STRUCT<
                move: STRUCT<name: STRING, url: STRING>,
                version_group_details: ARRAY<STRUCT<
                    level_learned_at: INT,
                    order: INT,
                    version_group: STRUCT<name: STRING, url: STRING>,
                    move_learn_method: STRUCT<name: STRING, url: STRING>
                >>
            >>'
        ) AS moves
    FROM ranked
    WHERE rn = 1
),
exploded_moves AS (
    SELECT
        p.pokemon_id,
        m.move.name AS move_name,
        m.version_group_details AS details
    FROM parsed p
    LATERAL VIEW EXPLODE(p.moves) move_lv AS m
),
exploded_details AS (
    SELECT
        m.pokemon_id,
        m.move_name,
        d.version_group.name AS version_group_name,
        d.move_learn_method.name AS learn_method_name,
        d.level_learned_at AS level_learned_at,
        d.order AS learn_order
    FROM exploded_moves m
    LATERAL VIEW EXPLODE(m.details) detail_lv AS d
)
SELECT
    COUNT(*) AS learning_detail_rows,
    COUNT(DISTINCT NAMED_STRUCT(
        'pokemon_id', pokemon_id,
        'move_name', move_name,
        'version_group_name', version_group_name,
        'learn_method_name', learn_method_name,
        'level_learned_at', level_learned_at,
        'learn_order', learn_order
    )) AS distinct_expanded_keys,
    SUM(
        CASE WHEN level_learned_at IS NULL THEN 1 ELSE 0 END
    ) AS null_levels,
    SUM(
        CASE WHEN learn_order IS NULL THEN 1 ELSE 0 END
    ) AS null_orders
FROM exploded_details;


In [0]:
WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.moves'),
            'ARRAY<STRUCT<move: STRUCT<name: STRING, url: STRING>>>'
        ) AS moves
    FROM ranked
    WHERE rn = 1
),
exploded AS (
    SELECT
        p.pokemon_id,
        m.move.name AS move_name,
        m.move.url AS move_url
    FROM parsed p
    LATERAL VIEW EXPLODE(p.moves) move_lv AS m
)
SELECT
    COUNT(*) AS move_rows,
    COUNT(DISTINCT NAMED_STRUCT(
        'pokemon_id', pokemon_id,
        'move_name', move_name
    )) AS distinct_pokemon_move_keys,
    SUM(CASE WHEN move_name IS NULL THEN 1 ELSE 0 END)
        AS null_move_names,
    SUM(CASE WHEN move_url IS NULL THEN 1 ELSE 0 END)
        AS null_move_urls
FROM exploded;

In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.forms'),
            'ARRAY<STRUCT<name: STRING, url: STRING>>'
        ) AS forms
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS pokemon_count,
    SUM(CASE WHEN forms IS NULL THEN 1 ELSE 0 END)
        AS null_form_arrays,
    SUM(CASE WHEN SIZE(forms) = 0 THEN 1 ELSE 0 END)
        AS empty_form_arrays,
    MIN(SIZE(forms)) AS min_forms,
    MAX(SIZE(forms)) AS max_forms,
    SUM(SIZE(forms)) AS total_form_references
FROM parsed;


In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.forms'),
            'ARRAY<STRUCT<name: STRING, url: STRING>>'
        ) AS forms
    FROM ranked
    WHERE rn = 1
),
exploded AS (
    SELECT
        p.pokemon_id,
        f.name AS form_name,
        f.url AS form_url
    FROM parsed p
    LATERAL VIEW EXPLODE(p.forms) form_lv AS f
)
SELECT
    COUNT(*) AS form_rows,
    COUNT(DISTINCT NAMED_STRUCT(
        'pokemon_id', pokemon_id,
        'form_name', form_name
    )) AS distinct_pokemon_form_keys,
    SUM(CASE WHEN form_name IS NULL THEN 1 ELSE 0 END)
        AS null_form_names,
    SUM(CASE WHEN form_url IS NULL THEN 1 ELSE 0 END)
        AS null_form_urls
FROM exploded;


In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.game_indices'),
            'ARRAY<STRUCT<
                game_index: INT,
                version: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS game_indices
    FROM ranked
    WHERE rn = 1
)
SELECT
    COUNT(*) AS pokemon_count,
    SUM(CASE WHEN game_indices IS NULL THEN 1 ELSE 0 END)
        AS null_game_index_arrays,
    SUM(CASE WHEN SIZE(game_indices) = 0 THEN 1 ELSE 0 END)
        AS empty_game_index_arrays,
    MIN(SIZE(game_indices)) AS min_game_indices,
    MAX(SIZE(game_indices)) AS max_game_indices,
    SUM(SIZE(game_indices)) AS total_game_index_rows
FROM parsed;


In [0]:

WITH ranked AS (
    SELECT
        raw_payload,
        ROW_NUMBER() OVER (
            PARTITION BY get_json_object(raw_payload, '$.id')
            ORDER BY _ingestion_timestamp DESC
        ) AS rn
    FROM pokemon.bronze.pokemon_raw
    WHERE status_code = 200
),
parsed AS (
    SELECT
        CAST(get_json_object(raw_payload, '$.id') AS INT)
            AS pokemon_id,
        from_json(
            get_json_object(raw_payload, '$.game_indices'),
            'ARRAY<STRUCT<
                game_index: INT,
                version: STRUCT<name: STRING, url: STRING>
            >>'
        ) AS game_indices
    FROM ranked
    WHERE rn = 1
),
exploded AS (
    SELECT
        p.pokemon_id,
        g.game_index,
        g.version.name AS version_name,
        g.version.url AS version_url
    FROM parsed p
    LATERAL VIEW EXPLODE(p.game_indices) game_lv AS g
)
SELECT
    COUNT(*) AS game_index_rows,
    COUNT(DISTINCT NAMED_STRUCT(
        'pokemon_id', pokemon_id,
        'version_name', version_name
    )) AS distinct_pokemon_version_keys,
    COUNT(DISTINCT NAMED_STRUCT(
        'pokemon_id', pokemon_id,
        'game_index', game_index,
        'version_name', version_name
    )) AS distinct_expanded_keys,
    SUM(CASE WHEN game_index IS NULL THEN 1 ELSE 0 END)
        AS null_game_indices,
    SUM(CASE WHEN version_name IS NULL THEN 1 ELSE 0 END)
        AS null_version_names,
    SUM(CASE WHEN version_url IS NULL THEN 1 ELSE 0 END)
        AS null_version_urls
FROM exploded;


In [0]:
SELECT pokemon_id
FROM pokemon.silver.pokemon
ORDER BY 1 

In [0]:
SHOW TABLES IN pokemon.gold;

In [0]:
--DESCRIBE TABLE pokemon.gold.dim_pokemon;
--DESCRIBE TABLE pokemon.gold.dim_type;
--DESCRIBE TABLE pokemon.gold.dim_ability;
--DESCRIBE TABLE pokemon.gold.dim_move;
--DESCRIBE TABLE pokemon.gold.dim_form;
--DESCRIBE TABLE pokemon.gold.dim_version_group;
--DESCRIBE TABLE pokemon.gold.dim_learn_method;
--DESCRIBE TABLE pokemon.gold.dim_game_version;
DESCRIBE TABLE pokemon.gold.dim_stat;

In [0]:
SELECT
    table_name,
    column_name,
    data_type,
    ordinal_position
FROM pokemon.information_schema.columns
WHERE table_schema = 'silver'
  AND table_name IN (
      'pokemon',
      'pokemon_types',
      'pokemon_abilities',
      'pokemon_stats',
      'pokemon_moves',
      'pokemon_move_details',
      'pokemon_forms',
      'pokemon_game_indices'
  )
ORDER BY table_name, ordinal_position;